# Simpler pipeline

Contrarily to TRASE-IN pipeline we rely only on GCaMP images. This supports 3D tracking & analysis.

We rely on Wavelet + SKT + EMC2 instead of StarDist + eMHT + EMC2 for tracking

This notebook has been developped under Ubuntu, with miniconda and python3.10.

For visualization, we mostly use `opencv` and `matplotlib` which gives us cross platform GUI. 

Sadly, we have noticed than GDK3 (the usual GUI backend used by opencv on Linux) seems to work better than the backend used for MacOS. With MacOs you will probably not have access to features like mouse zoom/drag in the displayed images.

Moreover, we decided to use Tk as a backend for interactive matplotlib (Tkinter should be installed by default in the conda env). We saw that it may fail with some installation/OS. In this case you can try to use the interactive inline backend ipymkl (pip install ipympl, replace %matplotlib tk with %matplotlib ipympl, and restart jupyter-notebook).

In [ ]:
import enum
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import tqdm

import byotrack
import byotrack.icy.io
import byotrack.fiji.io
from byotrack.implementation.refiner.interpolater import ForwardBackwardInterpolater
import byotrack.visualize

import trasein.detrending
import trasein.extraction
import trasein.filtering
import trasein.spike
import trasein.visualize

# Set TEST to True to reduce video sizes and computationnal burden 
TEST = False


# Set KEYS for visualization
# Defaults one are suited for AZERTY keyboards.
byotrack.visualize.InteractiveVisualizer.keys = {
    "QUIT": "q", # Quit
    "PAUSE": " ",# Pause/Unpause the video
    "LEFT": "w", # Go backward in time
    "RIGHT": "x",# Go forward in time
    "UP": "n",   # Go up through stacks (3D)
    "DOWN": "b", # Go down through stacks (3D)
    "DET": "d",  # Toggle detection mode
    "TRA": "t",  # Toggle track mode
    "VID": "v",  # Toggle video mode
}

## Loading videos

In [ ]:
# Set the path of the video data: You should choose the gcamp and tdtomato signal of the same video id
gcamp_path = "path/to/gcamp_video"

video = byotrack.Video(gcamp_path)

if TEST:
    video = video[:100]

# Normalize videos into [0, 1]
video.set_transform(
    byotrack.VideoTransformConfig(
        aggregate=True, normalize=True, q_min=0.02, q_max=0.9999, smooth_clip=1.0, compute_stats_on=10
    )
)

In [ ]:
# Display the video
# Use w/x to move forward in time (or space to run/pause the video)
# Use b/n to move through stacks (3D)
# Keys can be changed at the begining of the notebook

byotrack.visualize.InteractiveVisualizer(video).run()

## Track

### Loading already saved tracks

**This can be done only if you already have tracked neurons (and saved) on your video**

In [ ]:
# Reload tracks saved in the byotrack format

# Uncomment to run
# tracks = byotrack.Track.load("path/to/tracks.pt")

In [ ]:
# Or reload tracks from Icy xml format

# Uncomment to run
# tracks = byotrack.icy.io.load_tracks("path/to/tracks.xml")

In [ ]:
# Or reload tracks from ImageJ/Fiji xml format

# Uncomment to run
# tracks = byotrack.fiji.io.load_track("path/to/tracks.xml")

### Tracking pipeline

In [ ]:
from byotrack import BatchMultiStepTracker
from byotrack.implementation.detector.wavelet import WaveletDetector
from byotrack.implementation.linker.frame_by_frame.kalman_linker import KalmanLinker, KalmanLinkerParameters
from byotrack.implementation.refiner.cleaner import Cleaner
from byotrack.implementation.refiner.stitching.emc2 import EMC2Stitcher

#### Prepare detections (will be run online)

In [ ]:
# Parameters should be adapted to your data
# scale (int): Scale of objects. From 0 (smallest objects ~1px), to any positive integer
# k (float): Threshold to remove noise. The higher, the less objects are retrieved
#            Typical values between 1.0 and 10.0, default to 3.0
# min_area (float): Mininum number of pixels in an objects to be kept.

detector = WaveletDetector(scale=1, k=7, min_area=50, batch_size=1, device=torch.device("cpu"))

In [ ]:
# Display the detections with opencv (on 10 frames for debug purposes) 
# Use w/x to move forward in time (or space to run/pause the video)
# Use b/n to move through stacks (3D)
# Use v to switch on/off the display of the video
# Use d to switch detection display mode (None, mask, segmentation)
# Keys can be changed at the begining of the notebook

detections_sequence = detector.run(video[:5])
byotrack.visualize.InteractiveVisualizer(video, detections_sequence).run()

#### Detect & link into tracklets

In [ ]:
# Create the linker
# We set only the main parameters.
# You can look at the documentation to see the other parameters and more complete descriptions.

specs = KalmanLinkerParameters(
    association_threshold=10,  # Most important parameter: don't link if prediction and detection are at more than 10px away.
    detection_std=3.0,  # Detections location are precise up to 3.0 pixels (Usually ~ size of spots)
    process_std=1.5,  # Kalman filter predictions are precise up to 1.5 pixels (Usually ~ size of unmodeled displacement)
    kalman_order=0,  # Order of the kalman filter (0: Brownian, 1: Directed, 2: Accelerated, ...)
    cost="euclidean",
    track_building="smoothed",
    
)

linker = KalmanLinker(specs)

In [ ]:
# Run linking online (never loads the full video/detections in ram) 

tracker = BatchMultiStepTracker(detector, linker)

tracklets = tracker.run(video)

In [ ]:
# Visualize life span

byotrack.visualize.display_lifetime(tracklets)

#### Tracklet stitching

In [ ]:
# Cleaning + EMC2

cleaner = Cleaner(min_length=5, max_dist=3.5)
tracks = cleaner.run(video, tracklets)

stitcher = EMC2Stitcher(eta=5.0)  # Don't link tracks if they are too far (EMC dist > 5 (pixels))
tracks = stitcher.run(video, tracks)

In [ ]:
# Visualize new life span

byotrack.visualize.display_lifetime(tracks)

In [ ]:
# Save tracks

byotrack.Track.save(tracks, "tracks.pt")  # Can be reload with byotrack.Track.load("tracks.pt") (See above)

### Tracks visualization

You can export track to icy format and visualize them with icy, or use our own tool of visualization (or build new ones in python)

In [ ]:
# Export track to icy
# Needs to fill hole in tracks before saving with Forward backward interpolator

# Uncomment to run
# interpolater = ForwardBackwardInterpolater(method="constant", full=False)

# byotrack.icy.io.save_tracks(interpolater.run(tdtomato_video, tracks), "tracks.xml")

In [ ]:
byotrack.visualize.display_lifetime(tracks)

In [ ]:
# Display the tracks with opencv
# Use w/x to move forward in time (or space to run/pause the video)
# Use b/n to move through stacks (3D)
# Use v (resp. t) to switch on/off the display of video (resp. tracks)
# Keys can be changed at the begining of the notebook

byotrack.visualize.InteractiveVisualizer(video, tracks=tracks).run()

## Calcium signal extraction

### Select long tracks and complete them

In [ ]:
from byotrack.implementation.refiner.interpolater import ForwardBackwardInterpolater

# keep only big enough tracks (Cover at least 80% of video from start to end)

valid_tracks = [len(t) > 0.80 * len(video) for t in tracks]

interpolater = ForwardBackwardInterpolater(method="tps", full = True, alpha=10.0)
final_tracks = interpolater.run(video, tracks)  # Interpolate using all tracks, and filter afterwards
final_tracks = [track for i, track in enumerate(final_tracks) if valid_tracks[i]]

print(f"Kept {len(final_tracks)} valid tracks from {len(tracks)} tracks")

In [ ]:
# Sort & re id tracks

final_tracks = sorted(final_tracks, key=lambda track: float(track.points[0].sum()))
for i in range(len(final_tracks)):
    final_tracks[i].identifier = i

### GCaMP signal extraction

In [ ]:
# Extract control intensities from tdtomato sequence

raw_intensities = trasein.extraction.extract_intensities_from_roi(video, final_tracks, 9)

### Filtering noise signals

In [ ]:
# We first detrend using only frequency filtering (See detrending) as ICA creates some artefacts

detrended = trasein.detrending.high_pass_filter(raw_intensities, 1/100)

# Then test the Gaussian hypothesis. If rejected with less than thresh p_value, the signal is not noise
# Lower values of thresh => More noise

thresh = 1e-5

is_noise = trasein.filtering.is_noise(detrended, thresh)
print(f"Found {is_noise.sum()} noise signals")

In [ ]:
# Switch to non inline and interactive matplotlib
# It may fail on some installation (depending on what opencv/tkinter is relying on)
# Our more robust fix found is to rely on the interactive inline backend ipympl
# You can install it with pip install ipympl
# And use %matplotlib ipympl instead of %matplotlib tk
# Restart jupyter-notebook after installation.

%matplotlib tk

In [ ]:
# Interactive display of filtering
# Enable the correction of the filtering steps
# If the total size of the figure does not suits you, please adapt figsize to your need
# If sub figures are too small, you can adapt WIDTH/HEIGHT (number of subfigures by line/columns), less subfigures => bigger subfigures

title  = """Batch no {batch_id}/{MAX_BATCH}

Please use w/x to increase/decrease the batch id of signals displayed
Click on a signal to correct the filtering (Red signals are dropped, blue ones are kept)
"""


fig_size = (20, 20)
WIDTH = 4
HEIGHT = 4
MAX_BATCH = math.ceil(len(raw_intensities) / (WIDTH * HEIGHT))

batch_id = 0

fig, axs = plt.subplots(HEIGHT, WIDTH, sharex = 'col', sharey='row', figsize=fig_size)
colors = ("b", "r")


def plot():
    fig.suptitle(title.format(batch_id=batch_id, MAX_BATCH=MAX_BATCH))
    for i in range(HEIGHT):
        for j in range(WIDTH):
            k = batch_id * WIDTH * HEIGHT + i * WIDTH + j
            k = k % len(is_noise)
            axs[i, j].clear()
            axs[i, j].set_title("Rejected" if is_noise[k] else "Kept")
            axs[i, j].plot(raw_intensities[k] + 0.4, label="Raw intensity")
            axs[i, j].plot(detrended[k], color=colors[int(is_noise[k])], label="Detrended intensity")
            axs[i, j].legend()


def on_click(event):
    """Switch the noise status of signals on click"""
    for i in range(HEIGHT):
        for j in range(WIDTH):
            if axs[i, j] == event.inaxes:
                k = batch_id * WIDTH * HEIGHT + i * WIDTH + j
                k = k % len(is_noise)
                print(f"Manual switch of track {k}")
                is_noise[k] = not is_noise[k]

                # Replot the k
                axs[i, j].clear()
                axs[i, j].set_title("Rejected" if is_noise[k] else "Kept")
                axs[i, j].plot(raw_intensities[k] + 0.4, label="Raw intensity")
                axs[i, j].plot(detrended[k], color=colors[int(is_noise[k])], label="Detrended intensity")
                axs[i, j].legend()
                plt.draw()
                return


def on_press(event):
    """Change the batch id with w/x"""
    global batch_id

    if event.key in "wx":
        batch_id = (batch_id + (1 if event.key == "x" else -1)) % MAX_BATCH
        plot()
        fig.canvas.draw()

plot()

fig.canvas.mpl_connect('key_press_event', on_press)
fig.canvas.mpl_connect('button_press_event', on_click)

plt.show()

In [ ]:
# Re-Switch to inline matplotlib
%matplotlib inline

In [ ]:
# Optional manual filtering  (DO NOT WORK in 3D yet)
# Use w/x to move forward in time
# Use t to display none, valid, invalid or both tracks
# Double click to drop or keep a track for further analysis

print(f"There are {is_noise.sum()} noise signals")

interactive_filter = trasein.filtering.InteractiveTracksFiltering(video, tracks, ~is_noise)
interactive_filter.run()
is_noise = ~interactive_filter.is_valid

print(f"Updated to {is_noise.sum()} noise signals")

In [ ]:
# Re id final_tracks and calcium tracks when not noise so that display identifier matches with index

for i, k in enumerate(np.arange(len(final_tracks))[~is_noise]):
    final_tracks[k].identifier = i

### Detrending & Smoothing

In [ ]:
# Additional independent detrending to remove the remaining baseline
# Drop baseline (period larger than 100 frames)

detrended_intensities = trasein.detrending.high_pass_filter(raw_intensities[~is_noise], 1 / 100)

In [ ]:
# Smoothing with rolling average (the window size controls the amount of smoothing)

window_size = 5

calcium_signals = trasein.detrending.smooth(detrended_intensities, window_size)

In [ ]:
# Example of a particular neuron + 16 others
# You can select another n_id or batch_id

n_id = 0
batch_id = 0

plt.plot(raw_intensities[~is_noise][n_id] * 10 + 3, label="Raw intensities (Scaled)")
# plt.plot(ctrl_intensities[~is_noise][n_id] * 10 + 3, label="Ctrl intensities (Scaled)")
# plt.plot(corrected_intensities[n_id] - 5, label="ICA corrected (-5 offset)")
# plt.plot(detrended_intensities[n_id] - 5, label="Detrended (-5 offset)")
plt.plot(calcium_signals[n_id], label="Final smoothed & detrended signal")
plt.legend(loc="upper right", bbox_to_anchor=(1.65, 1))
plt.show()


fig, axs = plt.subplots(4, 4, sharex = 'col', sharey='row', figsize=(20, 20))

for i in range(4):
    for j in range(4):
        k = batch_id * 16 + i * 4 + j
        k = k % len(calcium_signals)

        axs[i, j].plot(raw_intensities[~is_noise][k] * 10 + 3)
        # axs[i, j].plot(ctrl_intensities[~is_noise][k] * 10 + 3)
        # axs[i, j].plot(corrected_intensities[k] - 5)
        # axs[i, j].plot(detrended_intensities[k] - 5)
        axs[i, j].plot(calcium_signals[k])
        

plt.show()

### Spike extraction

In [ ]:
# Apply foopsi to extract spikes and calcium signal reconstruction

calcium_reconstruction, spikes = trasein.spike.foopsi_all(calcium_signals)

In [ ]:
# Clusterize spikes
# You can use different std (how far the kernel will look for neighboring spikes to aggregate with the current one)
# 5 yields pretty good results

true_spikes = trasein.spike.clusterize_spikes(spikes, std=5.0)

In [ ]:
# Example to see how well the clustering has worked

plt.plot(spikes[n_id], label="Spikes")
plt.plot(true_spikes[n_id] + 0.25, label="Clustered (0.25 offset)")
plt.legend()
plt.show()

In [ ]:
# Visualize the spikes vs the signals
# You can select another n_id or batch_id

n_id = n_id
batch_id = 0

plt.plot(calcium_signals[n_id], label="Calcium signal")
plt.plot(calcium_reconstruction[n_id], label="Reconstruction")
# plt.plot(spikes[n_id] - 4, label="All spikes (-4 offset)")
plt.plot(true_spikes[n_id] - 2, label="Clustered spikes (-2 offset)")
plt.legend(loc="upper right", bbox_to_anchor=(1.65, 1))
plt.show()


fig, axs = plt.subplots(4, 4, sharex = 'col', sharey='row', figsize=(20, 20))

for i in range(4):
    for j in range(4):
        k = batch_id * 16 + i * 4 + j
        k = k % (len(calcium_signals))
        axs[i, j].plot(calcium_signals[k])
        axs[i, j].plot(calcium_reconstruction[k])
        # axs[i, j].plot(spikes[k] - 4)
        axs[i, j].plot(true_spikes[k] - 2)

plt.show()

In [ ]:
# Filter low spikes to keep only meaningful ones
# You can play with k to keep more or less spikes

k = 3

# First plot all spikes
plt.figure(figsize=(24, 16))
plt.title("All spikes")
plt.xlabel("Frames")
plt.ylabel("Neurons")
binarized_spikes = true_spikes > 0
pos = trasein.spike.to_raster_pos(binarized_spikes)
plt.eventplot(pos, linewidths=3)
plt.show()


# Then plot kept spikes
plt.figure(figsize=(24, 16))
plt.title("Kept spikes")
plt.xlabel("Frames")
plt.ylabel("Neurons")
binarized_spikes = trasein.spike.binarize_max_minus_std(true_spikes, k)

pos = trasein.spike.to_raster_pos(binarized_spikes)
plt.eventplot(pos, linewidths=3)
plt.show()

In [ ]:
# Visualize filtered and binarized spikes vs signals
# You can select another n_id or batch_id

n_id = n_id
batch_id = 0

plt.plot(raw_intensities[~is_noise][n_id] * 10 + 3, label="Raw intensities (Scaled)")
plt.plot(calcium_signals[n_id], label="Calcium signal")
plt.plot(binarized_spikes[n_id], label="Selected spikes")
plt.legend(loc="upper right", bbox_to_anchor=(1.65, 1))
plt.show()


fig, axs = plt.subplots(4, 4, sharex = 'col', sharey='row', figsize=(20, 20))

for i in range(4):
    for j in range(4):
        k = batch_id * 16 + i * 4 + j
        k = k % len(calcium_signals)
        axs[i, j].plot(raw_intensities[~is_noise][k] * 10 + 3)
        axs[i, j].plot(calcium_signals[k])
        axs[i, j].plot(binarized_spikes[k])

plt.show()

In [ ]:
# Visualize a particular neuron
# You can also follow this particular neuron on the interactive visualization to see where we fail and succeed

n_id = n_id  # Neuron id

print("Spiking frames:", np.arange(binarized_spikes.shape[1])[binarized_spikes[n_id] > 0])

plt.plot(raw_intensities[~is_noise][n_id] * 10 + 3, label="Raw intensities (Scaled)")
plt.plot(calcium_signals[n_id], label="Calcium signal")
plt.plot(binarized_spikes[n_id], label="Selected spikes")
plt.legend(loc="upper right", bbox_to_anchor=(1.65, 1))
plt.show()

# Interactive visualization of this neuron (on the video)
# Use w/x to move forward in time (or space to run/pause the video)
# Use t to display none, green, red or both tracks
# Use v to display none, green, red or both channels

k = np.arange(len(final_tracks))[~is_noise][n_id]

vis = byotrack.visualize.InteractiveVisualizer(video, tracks=final_tracks[k:k+1])

vis.scale = 1  # Increase/decrease the size of the display
vis._display_video = 3  # GCaMP

vis.run()